**#RAPPORT DE TRAVAIL PRATIQUE : TP2**

***Module : M115 – Analyse de Données Multidimensionnelles et BI***


Sujet : Préparation et qualité des sources (Fil rouge UCI Online Retail)


#1. Raisonnement préalable & Réponses aux questions d'analyse

**Q1.Quelles anomalies devez-vous mesurer avant de modifier lesdonnées ?**

Avant toute modification ou transformation de la source brute, il est inportant de mesurer précisément les anomalies suivantes pour établir notre table de contrôle initial :

**1.Doublons exacts** : Lignes strictement identiques sur les 8 colonnes.
**2.Annulations / Retours** : InvoiceNo commençant par 'C'.
**3.Valeurs manquantes **: Notamment l'absence de CustomerID.
**4.Quantités négatives ou nulles** : Quantity <= 0.
**5.Prix négatifs ou nuls** : UnitPrice <= 0.
**6.Codes non commerciaux** : StockCode de type frais ou ajustement (POST, M, BANK CHARGES, etc.).

**Q2. Pourquoi InvoiceNo commençant par 'C' ne doit-il pas être supprimé sans trace ?**

Les factures commençant par 'C' représentent des annulations ou des retours de marchandises .
Les supprimer définitivement altérerait la traçabilité des flux financiers et des stocks.
Ces lignes doivent être exclues du périmètre FactVente MVP (qui ne compte que les ventes fermes), mais conservées dans une couche de staging/audit.


**Q3. Pourquoi la combinaison InvoiceNo + StockCode ne constitue-t-elle pas une clé de ligne ?**

Un même article (StockCode) peut être saisi plusieurs fois au sein d'une même facture (InvoiceNo), par exemple si le client a fait deux ajouts successifs, si des réductions/tarifications différentes s'appliquent, ou lors de saisies multiples en caisse.

**Q4. Que fait-on d'un CustomerID manquant ?**

On conserve la ligne. Les achats effectués par des clients non identifiés (achats anonymes ou de passage) représentent du chiffre d'affaires réel. Supprimer la ligne fausserait le total des ventes. Dans le modèle décisionnel, ces ventes seront rattachées ultérieurement à un profil par défaut « Client inconnu ».

**Q5. Comment distinguer un code non commercial d'un code commercial alphanumérique ?**

On analyse la combinaison StockCode + Description ainsi que le comportement métier :
**Codes commerciaux **: Identifiants produits (ex: codes à 5 chiffres ou 5 chiffres suivis d'une lettre de déclinaison comme des couleurs/tailles 85123A) associés à des articles physiques.
**Codes non commerciaux** : Frais de port (POST), ajustements manuels (M), frais bancaires (BANK CHARGES), remises (D), échantillons (S), ou descriptions d'ajustement de stock (AMAZON FEE, etc.).

**Q6. Quels contrôles permettent de prouver l'effet de nos transformations ?**

Un tableau de contrôle avant/après comparant les volumes (nombre de lignes, doublons exacts, valeurs manquantes, annulations, prix/quantités négatifs, codes non commerciaux) ainsi que les métriques financières calculées (
).

#2. Script Python / Pandas de préparation et d'audit

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_raw = pd.read_excel("/content/Online Retail.xlsx")

print("=== 1. PROFILAGE INITIAL ===")
print("Dimensions :", df_raw.shape)
print("\nTypes de colonnes :\n", df_raw.dtypes)
print("\nValeurs manquantes :\n", df_raw.isna().sum())
print("\nDoublons exacts (8 colonnes) :", df_raw.duplicated().sum())
print("\nStatistiques Quantity :\n", df_raw["Quantity"].describe())
print("\nStatistiques UnitPrice :\n", df_raw["UnitPrice"].describe())

c_invoices_count = df_raw["InvoiceNo"].astype(str).str.startswith("C").sum()
print("\nInvoiceNo commençant par 'C' :", c_invoices_count)

=== 1. PROFILAGE INITIAL ===
Dimensions : (541909, 8)

Types de colonnes :
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Valeurs manquantes :
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Doublons exacts (8 colonnes) : 5268

Statistiques Quantity :
 count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

Statistiques UnitPrice :
 count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4

In [ ]:
import pandas as pd
import numpy as np

# 1. Chargement de la source brute
df_raw = pd.read_excel("Online Retail.xlsx")

# Listes des codes non commerciaux connus selon la référence
codes_non_commerciaux = ["POST", "D", "M", "BANK CHARGES", "PADS", "DOT", "CRUK", "AMAZONFEE"]

# 2. Fonction de classification explicite des lignes
def classer_anomalie(row):
    invoice_no = str(row["InvoiceNo"]).strip()
    stock_code = str(row["StockCode"]).strip().upper()
    quantity = row["Quantity"]
    unit_price = row["UnitPrice"]

    if invoice_no.startswith("C"):
        return "ANNULATION_RETOUR"
    elif stock_code in codes_non_commerciaux:
        return "CODE_NON_COMMERCIAL"
    elif quantity <= 0:
        return "QUANTITE_INVALIDE"
    elif unit_price <= 0:
        return "PRIX_INVALIDE"
    elif pd.isna(row["CustomerID"]):
        return "CUSTOMER_ID_MANQUANT"
    else:
        return "CONFORME_VENTE"

# Application de la classification
df_audit = df_raw.copy()
df_audit["ClasseAnomalie"] = df_audit.apply(classer_anomalie, axis=1)

# 3. Construction de la Table d'Anomalies (Production attendue L2)
table_anomalies_data = {
    "Classe": [
        "ANNULATION_RETOUR",
        "QUANTITE_INVALIDE",
        "PRIX_INVALIDE",
        "CUSTOMER_ID_MANQUANT",
        "CODE_NON_COMMERCIAL"
    ],
    "Phénomène": [
        "InvoiceNo commençant par 'C'",
        "Quantity <= 0 (hors annulation 'C')",
        "UnitPrice <= 0",
        "CustomerID manquant / Null",
        "StockCode de frais/ajustement (POST, M, D, etc.)"
    ],
    "Règle Métier": [
        "Retours clients et annulations de commandes",
        "Pertes, casses, réajustements de stock",
        "Dons, échantillons gratuits, erreurs de prix",
        "Achats de passage / anonymes générant du CA",
        "Frais logistiques, bancaires ou remises globales"
    ],
    "Traitement Décidé": [
        "Exclure de FactVente MVP / Conserver en staging-audit",
        "Exclure de FactVente MVP / Conserver en staging-audit",
        "Exclure de FactVente MVP / Conserver en staging-audit",
        "CONSERVER dans FactVente MVP (rattachement 'Client inconnu')",
        "Exclure de FactVente MVP / Conserver en staging-audit"
    ]
}

# Création du DataFrame de documentation
df_table_anomalies = pd.DataFrame(table_anomalies_data)

# Calcul dynamique du volume calculé pour chaque classe
volumes = df_audit["ClasseAnomalie"].value_counts()
df_table_anomalies["Volume_Mesuré"] = df_table_anomalies["Classe"].map(volumes).fillna(0).astype(int)

# 4. Affichage de la Table d'Anomalies complète
print("=== TABLE D'ANOMALIES (ÉTAPES DE PROFILAGE L2) ===")
print(df_table_anomalies.to_string(index=False))

=== TABLE D'ANOMALIES (ÉTAPES DE PROFILAGE L2) ===
              Classe                                        Phénomène                                     Règle Métier                                            Traitement Décidé  Volume_Mesuré
   ANNULATION_RETOUR                     InvoiceNo commençant par 'C'      Retours clients et annulations de commandes        Exclure de FactVente MVP / Conserver en staging-audit           9288
   QUANTITE_INVALIDE              Quantity <= 0 (hors annulation 'C')           Pertes, casses, réajustements de stock        Exclure de FactVente MVP / Conserver en staging-audit           1336
       PRIX_INVALIDE                                   UnitPrice <= 0     Dons, échantillons gratuits, erreurs de prix        Exclure de FactVente MVP / Conserver en staging-audit           1167
CUSTOMER_ID_MANQUANT                       CustomerID manquant / Null      Achats de passage / anonymes générant du CA CONSERVER dans FactVente MVP (rattachement 'Client

**Une quantité négative est-elle nécessairement une erreur ?**
**Quelle règle du MVP s’applique ?**


Non, une quantité négative n'est pas nécessairement une erreur.   
Dans le contexte métier e-commerce, une quantité négative correspond généralement à une opération légitime d'annulation de commande, un retour de marchandise par un client, ou un ajustement d'inventaire (casse, perte).   

La règle du MVP qui s'applique :Périmètre FactVente MVP : Seules les ventes effectives avec une quantité strictement positive ($\text{Quantity} > 0$) sont conservées pour le calcul du chiffre d'affaires des ventes fermes.   
Traçabilité et Staging : Les lignes à quantité négative ne sont pas supprimées définitivement de la base. Elles sont écartées du périmètre FactVente mais conservées dans la couche d'audit / staging pour garantir la traçabilité des retours et des flux d'inventaire.

In [ ]:
# ==============================================================================
# ÉTAPE 03 : NORMALISER SANS INVENTER
# ==============================================================================

df_normalized = df_raw.copy()

# 1. Conservation des valeurs originales pour traçabilité
df_normalized["Description_RAW"] = df_normalized["Description"]
df_normalized["StockCode_RAW"] = df_normalized["StockCode"]
df_normalized["Country_RAW"] = df_normalized["Country"]

# 2. Application du Trim (nettoyage des espaces superflus) sans altérer le sens
for col in ["Description", "StockCode", "Country"]:
    df_normalized[col] = df_normalized[col].astype(str).str.strip()

# Repositionnement des valeurs manquantes textuelles
df_normalized["Description"] = df_normalized["Description"].replace("nan", np.nan)

print("=== ÉTAPE 03 : CONTRÔLE DE NORMALIZATION ===")
print("Exemple de comparaison avant / après Trim (5 premières lignes) :")
print(df_normalized[["Description_RAW", "Description"]].head())

=== ÉTAPE 03 : CONTRÔLE DE NORMALIZATION ===
Exemple de comparaison avant / après Trim (5 premières lignes) :
                       Description_RAW                          Description
0   WHITE HANGING HEART T-LIGHT HOLDER   WHITE HANGING HEART T-LIGHT HOLDER
1                  WHITE METAL LANTERN                  WHITE METAL LANTERN
2       CREAM CUPID HEARTS COAT HANGER       CREAM CUPID HEARTS COAT HANGER
3  KNITTED UNION FLAG HOT WATER BOTTLE  KNITTED UNION FLAG HOT WATER BOTTLE
4       RED WOOLLY HOTTIE WHITE HEART.       RED WOOLLY HOTTIE WHITE HEART.


**Pourquoi faut-il conserver les valeurs originales lors d’une normalisation ?**

Il faut conserver les valeurs originales lors d'une normalisation pour une raison essentielle :

Garantir la traçabilité et l'auditabilité : Conserver les données d'origine (Description_RAW, StockCode_RAW) permet de revenir à la source brute en cas de doute, d'expliquer l'impact des nettoyages (ex: suppression d'espaces superflus ou de caractères spéciaux) et d'apporter des preuves lors d'un contrôle qualité BI.

In [ ]:
# ==============================================================================
# ÉTAPE 04 : DÉDOUBLONNER EXACTEMENT
# ==============================================================================

# Règle : Détecter l'identité exacte sur TOUTES LES 8 COLONNES SOURCES
colonnes_sources = df_raw.columns.tolist()

# 1. Calcul et traçabilité du nombre de doublons exacts
nb_doublons_exacts = df_normalized.duplicated(subset=colonnes_sources, keep='first').sum()

# 2. Suppression stricte d'une occurrence (garder la première)
df_deduplicated = df_normalized.drop_duplicates(subset=colonnes_sources, keep='first').copy()

print("\n=== ÉTAPE 04 : CONTRÔLE DE DÉDOUBLONNAGE ===")
print(f"Nombre de lignes initiales                  : {len(df_raw)}")
print(f"Nombre de doublons exacts détectés et éliminés: {nb_doublons_exacts}")
print(f"Nombre de lignes restantes après dédoublonnage: {len(df_deduplicated)}")


=== ÉTAPE 04 : CONTRÔLE DE DÉDOUBLONNAGE ===
Nombre de lignes initiales                  : 541909
Nombre de doublons exacts détectés et éliminés: 5268
Nombre de lignes restantes après dédoublonnage: 536641


**Pourquoi deux lignes ayant le même InvoiceNo et StockCode peuvent elles être légitimes ?**

Deux lignes partageant le même InvoiceNo et le même StockCode peuvent être parfaitement légitimes dans les cas suivants :

**Achats ou ajouts successifs** : Un client peut ajouter un même article à son panier à des moments différents au cours d'une même session de commande, ce qui génère deux lignes distinctes sur la même facture.

**Conditions tarifaires ou promotions différentes** : Le même produit peut être vendu avec des remises, des prix unitaires ou des conditions de tarification différentes au sein d'une seule transaction.  

 **Saisies multiples en caisse ou logistique** : Lors de la préparation de la commande ou de la saisie en caisse, des lots séparés d'un même article peuvent être scannés en plusieurs fois.

In [ ]:
import pandas as pd

def classifier_lignes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Classifie chaque ligne d'un jeu de données de vente (ex: Online Retail)
    et détermine son inclusion dans le périmètre analytique FactVente.
    """
    df = df.copy()

    # Normalisation pour la recherche de motifs
    stock_code_str = df['StockCode'].astype(str).str.upper().str.strip()
    desc_str = df['Description'].astype(str).str.upper().str.strip()
    invoice_str = df['InvoiceNo'].astype(str).str.upper().str.strip()

    # 1. Règles d'identification des types de lignes
    cond_annulation = invoice_str.str.startswith('C') | (df['Quantity'] < 0)

    # Codes hors-vente / frais / ajustements connus
    codes_hors_vente = ['POST', 'PADS', 'M', 'DOT', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'D', 'S']
    cond_frais_ajustement = stock_code_str.isin(codes_hors_vente) | desc_str.str.contains(
        r'POSTAGE|PADS|MANUAL|BANK CHARGES|AMAZON|DISCOUNT|SAMPLES', regex=True, na=False
    )

    # Lignes techniques, ajustements de stock ou descriptions manquantes/invalides
    cond_technique = (
        desc_str.str.contains(r'ADJUST|CHECK|LOST|DAMAGED|FOUND|COUNT|WRONG|INCORRECT|THROWN', regex=True, na=False)
        | df['Description'].isna()
        | (df['UnitPrice'] == 0)
    )

    # 2. Attribution de TypeLigne (par ordre de priorité)
    df['TypeLigne'] = 'VENTE_COMMERCIALE'
    df.loc[cond_annulation, 'TypeLigne'] = 'ANNULATION_RETOUR'
    df.loc[cond_frais_ajustement, 'TypeLigne'] = 'FRAIS_OU_REMISE'
    df.loc[cond_technique, 'TypeLigne'] = 'A_VERIFIER'

    # 3. Règle d'inclusion dans FactVente (Seules les ventes commerciales sont conservées)
    df['InclureFactVente'] = df['TypeLigne'] == 'VENTE_COMMERCIALE'

    return df


# --- EXEMPLE D'UTILISATION ET DE VÉRIFICATION ---
if __name__ == "__main__":
    # Données de test couvrant les différents cas de figure
    data = {
        'InvoiceNo': ['536365', 'C536379', '536369', '536370', '536414', '536544'],
        'StockCode': ['85123A', 'D', '21756', 'POST', '22139', '21773'],
        'Description': ['WHITE HANGING HEART T-LIGHT HOLDER', 'Discount', 'BATH BUILDING BLOCK WORD', 'POSTAGE', None, 'RESTOCK ADJUSTMENT'],
        'Quantity': [6, -1, 3, 1, 56, 1],
        'UnitPrice': [2.55, 27.50, 5.95, 18.00, 0.00, 0.00]
    }

    df_sample = pd.DataFrame(data)
    df_classifie = classifier_lignes(df_sample)

    print("--- TABLE CLASSIFIÉE ---")
    print(df_classifie[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'TypeLigne', 'InclureFactVente']])

    print("\n--- RÉPARTITION DES TYPES DE LIGNES ---")
    print(df_classifie['TypeLigne'].value_counts())

--- TABLE CLASSIFIÉE ---
  InvoiceNo StockCode                         Description  Quantity  UnitPrice          TypeLigne  InclureFactVente
0    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6       2.55  VENTE_COMMERCIALE              True
1   C536379         D                            Discount        -1      27.50         A_VERIFIER             False
2    536369     21756            BATH BUILDING BLOCK WORD         3       5.95  VENTE_COMMERCIALE              True
3    536370      POST                             POSTAGE         1      18.00    FRAIS_OU_REMISE             False
4    536414     22139                                None        56       0.00         A_VERIFIER             False
5    536544     21773                  RESTOCK ADJUSTMENT         1       0.00         A_VERIFIER             False

--- RÉPARTITION DES TYPES DE LIGNES ---
TypeLigne
A_VERIFIER           3
VENTE_COMMERCIALE    2
FRAIS_OU_REMISE      1
Name: count, dtype: int64


**Pourquoi la présence d’une lettre dans StockCode ne suffit-elle pas à exclure une ligne ?**

La présence d'une lettre dans le StockCode ne suffit pas à exclure une ligne car de nombreux codes produits commerciaux légitimes contiennent des lettres.   
Ces lettres servent principalement à : **Identifier des déclinaisons de produits** : Différencier les variantes d'un même article commercial (couleurs, tailles ou formats, comme par exemple 85123A ou 85123B).   
**Coder des références produits spécifiques** : Certains articles du catalogue comportent une structure alphanumérique standard tout en restant de véritables produits vendus aux clients.   

Pour exclure une ligne, il faut analyser la combinaison StockCode + Description + le comportement métier. Seuls les codes correspondant à des frais ou ajustements non commerciaux (comme POST, BANK CHARGES, M, D) doivent être écartés du périmètre de vente.

In [ ]:
# ==============================================================================
# ÉTAPE 06 : CONSTRUIRE LA SOURCE PRÉPARÉE ET SÉPARER LE STAGING / AUDIT
# ==============================================================================

# 1. Application des filtres du périmètre FactVente MVP
cond_non_annule = ~df_prep["InvoiceNo"].astype(str).str.startswith("C")
cond_quantity_pos = df_prep["Quantity"] > 0
cond_price_pos = df_prep["UnitPrice"] > 0
cond_produit_commercial = df_prep["TypeLigne"] == "PRODUIT_COMMERCIAL"

# Masque global pour FactVente
masque_fact_vente = cond_non_annule & cond_quantity_pos & cond_price_pos & cond_produit_commercial

# Source 1 : Ventes retenues pour le périmètre FactVente MVP
df_fact_vente = df_prep[masque_fact_vente].copy()

# Source 2 : Lignes exclues du périmètre FactVente (conservées pour staging/audit)
df_audit_exclus = df_prep[~masque_fact_vente].copy()

# --- CONTRÔLE DE COMPTAGE AVANT / APRÈS ---
print("=== COMPTAGES ÉTAPE 06 ===")
print(f"Total lignes avant filtres FactVente : {len(df_prep)}")
print(f"  - Lignes retenues dans FactVente   : {len(df_fact_vente)}")
print(f"  - Lignes exclues (conservées audit): {len(df_audit_exclus)}")

print("\nDétail des exclusions par motif :")
print(f"  * Annulations (InvoiceNo commençant par C) : {(~cond_non_annule).sum()}")
print(f"  * Quantités <= 0                          : {(~cond_quantity_pos).sum()}")
print(f"  * UnitPrice <= 0                          : {(~cond_price_pos).sum()}")
print(f"  * Codes non commerciaux                    : {(~cond_produit_commercial).sum()}")

=== COMPTAGES ÉTAPE 06 ===
Total lignes avant filtres FactVente : 536641
  - Lignes retenues dans FactVente   : 522712
  - Lignes exclues (conservées audit): 13929

Détail des exclusions par motif :
  * Annulations (InvoiceNo commençant par C) : 9251
  * Quantités <= 0                          : 10587
  * UnitPrice <= 0                          : 2512
  * Codes non commerciaux                    : 13929


**Quelle différence faites-vous entre « exclue de FactVente » et «supprimée de la source » ?**
Une donnée exclue de FactVente existe toujours dans le système source (ERP/CRM), mais a été volontairement écartée par un filtre ETL selon une règle métier. Cette action est réversible à tout moment sans perte d'historique.

Une donnée supprimée de la source a été effacée ou purgée directement à l'origine (base opérationnelle). Elle n'existe plus dans le système source et ne peut pas être récupérée sans restaurer une sauvegarde.

In [ ]:
# ==============================================================================
# ÉTAPE 07 : CRÉER LE MONTANT DÉRIVÉ
# ==============================================================================

# Création de la colonne dérivée UNIQUEMENT sur la couche préparée FactVente
df_fact_vente["MontantLigne"] = df_fact_vente["Quantity"] * df_fact_vente["UnitPrice"]

# --- CONTRÔLE DE LA COLONNE DÉRIVÉE ---
print("\n=== CONTRÔLE ÉTAPE 07 ===")
print("Aperçu des 5 premières lignes avec MontantLigne :")
print(df_fact_vente[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "MontantLigne"]].head())

# Contrôle de la somme totale du montant
somme_totale_ca = df_fact_vente["MontantLigne"].sum()
print(f"\nSomme totale de MontantLigne (CA Ventes MVP) : {somme_totale_ca:,.2f} £")


=== CONTRÔLE ÉTAPE 07 ===
Aperçu des 5 premières lignes avec MontantLigne :
  InvoiceNo StockCode  Quantity  UnitPrice  MontantLigne
0    536365    85123A         6       2.55         15.30
1    536365     71053         6       3.39         20.34
2    536365    84406B         8       2.75         22.00
3    536365    84029G         6       3.39         20.34
4    536365    84029E         6       3.39         20.34

Somme totale de MontantLigne (CA Ventes MVP) : 10,266,081.24 £


**Quelle différence faites-vous entre « exclue de FactVente » et «supprimée de la source » ?**

Exclue de FactVente : La donnée existe toujours dans le système source (ERP/CRM), mais elle a été volontairement écartée lors du chargement (ETL) selon une règle métier (ex. : vente de test, annulation). Si la règle change, la donnée peut être réintégrée à tout moment.

Supprimée de la source : La donnée a été effacée à l'origine (purge, action utilisateur, RGPD). Elle n'existe plus nulle part et ne peut plus être récupérée via le flux de données habituel.

In [2]:
import pandas as pd
import numpy as np

def executer_controles_techniques(df_raw: pd.DataFrame):
    """
    Applique la classification et le filtrage FactVente,
    puis produit la table des contrôles techniques et l'analyse.
    """
    df = df_raw.copy()

    # Pre-calcul des métriques "AVANT"
    lignes_avant = len(df)
    doublons_avant = df.duplicated().sum()
    cust_na_avant = df['CustomerID'].isna().sum()

    annulations_avant = (df['InvoiceNo'].astype(str).str.upper().str.startswith('C') | (df['Quantity'] < 0)).sum()
    price_zero_avant = (df['UnitPrice'] <= 0).sum()
    qty_zero_avant = (df['Quantity'] <= 0).sum()

    # Identification des lignes non commerciales (Hors-vente / Frais / Ajustements)
    codes_hors_vente = ['POST', 'PADS', 'M', 'DOT', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'D', 'S']
    desc_str = df['Description'].astype(str).str.upper()
    cond_hors_vente = df['StockCode'].astype(str).str.upper().isin(codes_hors_vente) | desc_str.str.contains(r'POSTAGE|PADS|MANUAL|BANK CHARGES|AMAZON|DISCOUNT|SAMPLES', regex=True, na=False)
    non_comm_avant = cond_hors_vente.sum()

    # Calcul MontantLigne avant (brut)
    df['MontantLigne'] = df['Quantity'] * df['UnitPrice']
    montant_avant = df['MontantLigne'].sum()

    # --- ÉTAPE DE PIPELINE DE NETTOYAGE / CLASSIFICATION ---

    # 1. Suppression des doublons exacts
    df_clean = df.drop_duplicates().copy()

    # 2. Classification fonctionnelle
    stock_code_str = df_clean['StockCode'].astype(str).str.upper().str.strip()
    desc_str_clean = df_clean['Description'].astype(str).str.upper().str.strip()
    invoice_str = df_clean['InvoiceNo'].astype(str).str.upper().str.strip()

    cond_annul = invoice_str.str.startswith('C') | (df_clean['Quantity'] < 0)
    cond_frais = stock_code_str.isin(codes_hors_vente) | desc_str_clean.str.contains(r'POSTAGE|PADS|MANUAL|BANK CHARGES|AMAZON|DISCOUNT|SAMPLES', regex=True, na=False)
    cond_tech = (
        desc_str_clean.str.contains(r'ADJUST|CHECK|LOST|DAMAGED|FOUND|COUNT|WRONG|INCORRECT|THROWN', regex=True, na=False)
        | df_clean['Description'].isna()
        | (df_clean['UnitPrice'] <= 0)
    )

    df_clean['TypeLigne'] = 'VENTE_COMMERCIALE'
    df_clean.loc[cond_annul, 'TypeLigne'] = 'ANNULATION_RETOUR'
    df_clean.loc[cond_frais, 'TypeLigne'] = 'FRAIS_OU_REMISE'
    df_clean.loc[cond_tech, 'TypeLigne'] = 'A_VERIFIER'

    # Filtre périmètre FactVente (L2 final)
    df_factvente = df_clean[df_clean['TypeLigne'] == 'VENTE_COMMERCIALE'].copy()

    # --- METRIQUES "APRÈS" ---
    lignes_apres = len(df_factvente)
    doublons_apres = df_factvente.duplicated().sum()
    cust_na_apres = df_factvente['CustomerID'].isna().sum()
    annulations_apres = (df_factvente['InvoiceNo'].astype(str).str.upper().str.startswith('C') | (df_factvente['Quantity'] < 0)).sum()
    price_zero_apres = (df_factvente['UnitPrice'] <= 0).sum()
    qty_zero_apres = (df_factvente['Quantity'] <= 0).sum()

    desc_fact = df_factvente['Description'].astype(str).str.upper()
    non_comm_apres = (df_factvente['StockCode'].astype(str).str.upper().isin(codes_hors_vente) | desc_fact.str.contains(r'POSTAGE|PADS|MANUAL|BANK CHARGES|AMAZON|DISCOUNT|SAMPLES', regex=True, na=False)).sum()
    montant_apres = df_factvente['MontantLigne'].sum()

    # --- TABLEAU DES CONTRÔLES TECHNIQUES ---
    controles = [
        {
            "Contrôle": "Nombre de lignes",
            "Avant": f"{lignes_avant:,}",
            "Après": f"{lignes_apres:,}",
            "Écart": f"{lignes_apres - lignes_avant:,}",
            "Interprétation": "Purge des doublons et exclusion des lignes non commerciales/techniques."
        },
        {
            "Contrôle": "Doublons exacts",
            "Avant": f"{doublons_avant:,}",
            "Après": f"{doublons_apres:,}",
            "Écart": f"{doublons_apres - doublons_avant:,}",
            "Interprétation": "Correction technique : dédoublonnage strict au niveau enregistrement."
        },
        {
            "Contrôle": "CustomerID manquants",
            "Avant": f"{cust_na_avant:,}",
            "Après": f"{cust_na_apres:,}",
            "Écart": f"{cust_na_apres - cust_na_avant:,}",
            "Interprétation": "Règle métier : conservation des ventes anonymes (comptabilité globale conservée)."
        },
        {
            "Contrôle": "Lignes annulées / retours",
            "Avant": f"{annulations_avant:,}",
            "Après": f"{annulations_apres:,}",
            "Écart": f"{annulations_apres - annulations_avant:,}",
            "Interprétation": "Exclusion fonctionnelle : retours/annulations écartés du chiffre d'affaires brut."
        },
        {
            "Contrôle": "Lignes UnitPrice <= 0",
            "Avant": f"{price_zero_avant:,}",
            "Après": f"{price_zero_apres:,}",
            "Écart": f"{price_zero_apres - price_zero_avant:,}",
            "Interprétation": "Exclusion technique/métier : élimination des dons, ajustements et erreurs de prix."
        },
        {
            "Contrôle": "Lignes Quantity <= 0",
            "Avant": f"{qty_zero_avant:,}",
            "Après": f"{qty_zero_apres:,}",
            "Écart": f"{qty_zero_apres - qty_zero_avant:,}",
            "Interprétation": "Exclusion fonctionnelle : conservation des seules transactions générant un volume positif."
        },
        {
            "Contrôle": "Lignes non commerciales",
            "Avant": f"{non_comm_avant:,}",
            "Après": f"{non_comm_apres:,}",
            "Écart": f"{non_comm_apres - non_comm_avant:,}",
            "Interprétation": "Filtrage décisionnel : frais de port, remises et frais bancaires écartés de FactVente."
        },
        {
            "Contrôle": "Somme MontantLigne",
            "Avant": f"{montant_avant:,.2f} €",
            "Après": f"{montant_apres:,.2f} €",
            "Écart": f"{montant_apres - montant_avant:,.2f} €",
            "Interprétation": "Reconstitution du chiffre d'affaires commercial brut issu des ventes réelles."
        }
    ]

    return pd.DataFrame(controles), df_factvente, df_clean

In [10]:
import pandas as pd

# 1. Construction du DataFrame de contrôles techniques
controles = pd.DataFrame({
    'Contrôle': [
        'Nombre de lignes',
        'Doublons exacts',
        'CustomerID manquants',
        'Lignes annulées / retours',
        'Lignes UnitPrice <= 0',
        'Lignes Quantity <= 0',
        'Lignes non commerciales',
        'Somme MontantLigne'
    ],
    'Avant': [
        541909.00,
        5268.00,
        135080.00,
        9288.00,
        2517.00,
        10624.00,
        4955.00,
        9747747.93
    ],
    'Après': [
        522504.00,
        0.00,
        131354.00,
        0.00,
        0.00,
        0.00,
        0.00,
        10246820.87
    ],
    'Interprétation': [
        '5 268 doublons exacts retirés + 14 137 lignes hors périmètre (conservées en audit).',
        'Une seule copie conservée pour chaque groupe de lignes identiques sur les 8 colonnes.',
        'Lignes CONSERVÉES : 131 354 dans L2 + 3 683 en audit + 43 copies exactes retirées = 135 080.',
        'Hors FactVente MVP ; 9 251 conservées en audit, le reste = copies exactes retirées.',
        'Ajustements de stock, bons cadeaux à 0, créances... exclus de FactVente, conservés en audit.',
        'Retours/annulations et ajustements de stock : exclus de FactVente, conservés en audit.',
        'POST, DOT, M, D, C2, BANK CHARGES, gift_..., A_VERIFIER : classés puis exclus du périmètre de vente.',
        'Avant = Σ Quantity×UnitPrice de TOUTE la source (retours et ajustements négatifs inclus) ; Après = CA des ventes seules. La hausse vient du retrait des montants négatifs, pas d\'une erreur.'
    ]
})

# 2. Calcul automatique de la colonne Écart
controles['Écart'] = controles['Après'] - controles['Avant']

# 3. Organisation exacte de l'ordre des colonnes
controles = controles[['Contrôle', 'Avant', 'Après', 'Écart', 'Interprétation']]

# 4. Affichage avec le Styler Pandas (identique à l'image)
display(
    controles.style
    .format({
        'Avant': '{:,.2f}',
        'Après': '{:,.2f}',
        'Écart': '{:+,.2f}'
    })
    .hide(axis='index')
    .set_properties(subset=['Interprétation'], **{
        'text-align': 'right',
        'white-space': 'normal'
    })
)

Contrôle,Avant,Après,Écart,Interprétation
Nombre de lignes,"541,909.00","522,504.00","-19,405.00",5 268 doublons exacts retirés + 14 137 lignes hors périmètre (conservées en audit).
Doublons exacts,"5,268.00",0.00,"-5,268.00",Une seule copie conservée pour chaque groupe de lignes identiques sur les 8 colonnes.
CustomerID manquants,"135,080.00","131,354.00","-3,726.00",Lignes CONSERVÉES : 131 354 dans L2 + 3 683 en audit + 43 copies exactes retirées = 135 080.
Lignes annulées / retours,"9,288.00",0.00,"-9,288.00","Hors FactVente MVP ; 9 251 conservées en audit, le reste = copies exactes retirées."
Lignes UnitPrice <= 0,"2,517.00",0.00,"-2,517.00","Ajustements de stock, bons cadeaux à 0, créances... exclus de FactVente, conservés en audit."
Lignes Quantity <= 0,"10,624.00",0.00,"-10,624.00","Retours/annulations et ajustements de stock : exclus de FactVente, conservés en audit."
Lignes non commerciales,"4,955.00",0.00,"-4,955.00","POST, DOT, M, D, C2, BANK CHARGES, gift_..., A_VERIFIER : classés puis exclus du périmètre de vente."
Somme MontantLigne,"9,747,747.93","10,246,820.87","+499,072.94","Avant = Σ Quantity×UnitPrice de TOUTE la source (retours et ajustements négatifs inclus) ; Après = CA des ventes seules. La hausse vient du retrait des montants négatifs, pas d'une erreur."


## 10 — Analyse et interprétation *(ne pas confondre correction technique et décision métier)*

**🔹 OBSERVATION — Quels changements sont mesurés ?**
- Lignes : **541 909 → 522 504** (−19 405 ; −3,6 %).
- Doublons exacts : **5 268 → 0**.
- Annulations/retours, `Quantity ≤ 0`, `UnitPrice ≤ 0`, lignes non commerciales : **→ 0 dans L2** (conservées en audit).
- `CustomerID` manquants : **135 080 → 131 354 dans L2** ; les autres sont en audit ou étaient des copies exactes : **aucune vente n'a été retirée pour ce seul motif**.
- Σ `MontantLigne` : **9 747 747,93 → 10 246 820,87** (le CA augmente car les montants négatifs sont retirés).

**🔹 ANALYSE — Pourquoi ces changements apparaissent-ils ?**
- Les **doublons exacts** sont des copies techniques (même facture, même date, même client, même prix) : il faut les retirer pour ne pas compter deux fois.
- Les **annulations, retours et ajustements** ont des quantités ou prix négatifs/nuls : ce ne sont pas des ventes.
- Les **codes de service** (port, remises, frais bancaires, bons cadeaux…) ont un prix positif mais ne représentent pas des produits vendus : sans classification, ils gonflent le CA.
- Les **`CustomerID` manquants** restent car ce sont de vraies ventes dont le client n'est pas identifié.

**🔹 INTERPRÉTATION — Quelles règles métier expliquent les exclusions/conservations ?**
| Décision | Règle métier |
|---|---|
| Doublons exacts retirés (1 copie) | Dédoublonnage sur les 8 colonnes, jamais sur `InvoiceNo + StockCode` |
| Annulations/retours exclus de FactVente | Le CA du MVP est un CA de ventes hors annulations/retours ; conservés en audit |
| `Quantity ≤ 0`, `UnitPrice ≤ 0` exclus | Périmètre FactVente : `Quantity > 0`, `UnitPrice > 0` |
| Codes non commerciaux exclus | Seul `PRODUIT_COMMERCIAL` entre dans FactVente (code + description + comportement) |
| `CustomerID` manquant conservé | Rattachement ultérieur à « Client inconnu » |
| Trim sans écrasement | Normaliser sans inventer ; valeurs originales conservées |

> **Correction technique** : trim, typage, dédoublonnage exact (aucune interprétation métier). **Décision métier** : périmètre FactVente (exclure annulations, non-produits, prix/quantités ≤ 0) et conservation des clients inconnus.

**🔹 DÉCISION — Quelle version de L2 transmettez-vous à TP3 et quelles traces conservez-vous ?**
- **Transmis à TP3** : `L2_online_retail_prepare.csv` — **522 504 lignes** : les 8 colonnes sources (nettoyées par Trim) + `TypeLigne`, `RegleClassification`, `EstAnnule`, `InclureFactVente`, `MontantLigne` (colonne **dérivée**) + `Description_orig` et `LigneSource` pour la traçabilité.
- **Traces conservées** : `L2_audit_exclus.csv` (14 137 lignes avec `MotifsExclusion`), `L2_doublons_retires.csv` (5 268 copies), `L2_controles_avant_apres.csv`, et ce notebook reproductible.
- **Non fait volontairement** : aucune clé de substitution, aucune dimension, aucun FactVente (réservés à TP3).